# Guia sobre o modelo utilizado

Guia de consulta dos notebooks que precisam ser executados, **nesta ordem**, para tratar os
dados e chegar à aplicação do modelo final do projeto Ignis.

Este notebook **não executa nada do pipeline**: ele apenas documenta. Todo o conteúdo abaixo
foi extraído da leitura das células de código e de markdown dos notebooks reais do
repositório. Onde o código não permite afirmar algo, consta **"não identificado"**.

Quem quiser **ver o modelo final rodando**, e não apenas descrito, deve abrir
`notebooks/modelo_final/modelo_final.ipynb`, que reproduz a formulação entregue, mede o erro na
validação cruzada e no conjunto de teste reservado, e mostra um exemplo de triagem.

O Ignis é um **modelo substituto da simulação térmica** de fogões de piso: prevê a
temperatura de cada ponto de medição (termopar) a partir das decisões de projeto do ensaio,
sem rodar a simulação CFD completa. O pipeline parte da planilha bruta de ensaios da
parceira, reconstrói a tabela, padroniza categorias, divide a base em **sete conjuntos, um
por área térmica**, trata valores ausentes, treina e compara estimadores sob um protocolo de
validação fixo, e chega à **formulação entregue** (a receita que o dashboard reproduz em
`src/frontend/dados/exportar.py`).

A previsão de cada componente é confrontada com o seu **limite de segurança** (`active
limit`), de modo que o uso pretendido é de **triagem**: apontar quais componentes
provavelmente estouram o limite, para dirigir o ensaio físico a eles.

> **Caminhos.** Todos os caminhos citados são relativos à **raiz do repositório**
> (por exemplo `notebooks/colabs_S4/...`), e não à pasta deste guia.
>
> **Nomes de colunas e de algoritmos.** Ficam no original em inglês, e os nomes de colunas
> aparecem em formato de código. As duas colunas mais citadas são `reason`, o motivo declarado
> do ensaio, e `losses`, o perfil de perdas imposto a ele.


## Sumário

- [Diagrama de fluxo](#diagrama-de-fluxo)
- [Notebooks](#notebooks)
- [Saídas sem consumidor](#saídas-sem-consumidor)
- [Arquivos externos ao pipeline que são necessários](#arquivos-externos-ao-pipeline-que-são-necessários)
- [Célula opcional de verificação](#célula-opcional-de-verificação)
- [Conclusão sobre o modelo final utilizado](#conclusão-sobre-o-modelo-final-utilizado)
- [Referências](#referências)


## Diagrama de fluxo

<img src="../../documents/assets/modeloFinal/fluxo_pipeline.png" width="720" alt="Fluxo do pipeline Ignis, do dataset_original.xlsx ate o dashboard">

<sub>**Azul**: a cadeia principal, na ordem de execução. **Cinza tracejado**: arquivo externo, que o pipeline consome mas não produz. **Cinza sólido**: o ramo lateral, cujos `_snull` ninguém lê adiante. **Verde**: treino e aplicação, em que os quatro notebooks independem uns dos outros; a seta que sai do bloco é conceitual, e não um arquivo.</sub>

Os sete notebooks abrem com a mesma célula de caminhos: `raiz_repositorio()` sobe os
diretórios até achar a pasta que contém `notebooks/` e `documents/`, e depois um `os.chdir`
entra em `notebooks/colabs_S4`. É por isso que os CSVs intermediários são lidos e gravados
nessa pasta, mesmo quando o código os cita só pelo nome.

Em resumo, a cadeia de arquivos é `01`, depois `02`, e depois qualquer um entre `04`, `05`,
`06` e `07`. O `03` fica fora dessa cadeia.


---

## Notebooks

Os sete notebooks a seguir, na ordem em que precisam ser executados, e por último o script que
transforma o modelo escolhido em arquivo. Cada seção traz o objetivo, os arquivos lidos, as
etapas, as funções definidas e os arquivos gravados.


---

### 01. `preparacao_datasets`

**Caminho real:** `notebooks/colabs_S4/preparacao_datasets.ipynb`

<sub>bibliotecas: pandas, numpy, openpyxl (indireta, exigida por pandas para ler .xlsx)</sub>

#### Objetivo

Primeiro notebook do pipeline. A planilha da parceira vem **transposta**: os ensaios estão
nas colunas e os componentes nas linhas, o inverso do que um modelo espera. Este notebook
reconstrói a estrutura, padroniza as categorias escritas de formas diferentes, extrai
atributos físicos do texto da coluna `reason` e divide a base em **sete conjuntos, um por
área térmica**, no formato *wide*. Existe neste ponto porque nenhum outro notebook lê o Excel:
todos os seguintes partem dos CSVs daqui.

Descarta `Reason`, `TC Number` e `Key Area` das features. O `Active Limit` sai do treino e é
salvo à parte como tabela de referência. Outliers, nulos e one-hot são etapas posteriores.

#### Arquivos de entrada

| Arquivo | Formato | Origem |
|---|---|---|
| `documents/outros/dataset_original.xlsx` | Excel, aba 0, lido com `header=None` | Fonte externa, fornecida pela parceira. O código tenta uma lista de caminhos candidatos (`_CANDIDATOS`) e adota o primeiro que existir: `dataset_original.xlsx`, `data/dataset_original.xlsx`, `/content/dataset_original.xlsx` (Colab), `../../documents/outros/...`, `../documents/outros/...`, `documents/outros/...`. No repositório, o que resolve é `documents/outros/dataset_original.xlsx`. |

A planilha crua é descrita no markdown como **97 linhas por 60 colunas**.

#### Etapas principais

1. Lê a planilha sem cabeçalho, porque o arquivo vem transposto: ensaios nas colunas e
   componentes nas linhas.
2. Reconstrói as sete variáveis de configuração a partir das linhas 2 a 8 e numera cada
   ensaio em `run_idx`, já que os nomes originais se repetem.
3. Padroniza as categorias escritas de formas diferentes e unifica `housing`, cuja diferença
   entre os dois rótulos inverte de sinal conforme o `run_case`.
4. Extrai do texto de `reason` a família do ensaio, em seis níveis exclusivos, e do rótulo de
   `losses` a ordem de chaveamento dos IGBTs.
5. Separa os componentes, recupera a área por preenchimento para baixo e extrai o limite
   numérico de cada um.
6. Converte ao formato longo, junta as configurações e divide em sete conjuntos por área,
   descartando componentes idênticos bit a bit.
7. Grava a chave de replicata, usada apenas para estimar o piso de ruído.

**Resultado.** A última célula imprime uma linha por área, com o arquivo gravado, o número de
ensaios e o número de componentes. A conferência do suporte mostra quais indicadores derivados de `reason`
foram mantidos e quais foram descartados por aparecerem em menos de três ensaios.

#### Funções definidas

Agrupadas pelo papel que exercem.

**Caminho do projeto**

- `raiz_repositorio(inicio=None)`: sobe os diretórios até achar a raiz, para que o notebook
  rode a partir de qualquer pasta. Devolve o caminho, ou `None`.

**Padronização de texto**

- `normalizar(texto)`: minúsculas, sem acento e sem espaço duplo, de modo que duas grafias da
  mesma categoria deixem de ser categorias distintas.
- `slug(texto)`: converte o nome da área em nome de arquivo.

**Leitura de `reason`**

- `familia_do_motivo(t)`: classifica o motivo em um de seis níveis mutuamente exclusivos, por
  ordem de precedência, prevalecendo o caminho do ar sobre o material.
- `declara_vazao(t)` e `vazao_alta(t)`: indicam se o motivo menciona a intensidade do soprador
  e se o nível declarado é o mais alto.
- `suporte_por_nivel(serie)`: conta os ensaios do nível menos representado, critério que
  decide se um indicador entra ou é descartado.
- `_marcador_doe(pos)`: acrescenta o índice do planejamento de experimentos à chave de
  replicata, para que dois pontos distintos não passem por repetição.

**Rótulo de `losses`**

- `ordem_de_acionamento(rotulo)`: separa a ordem de chaveamento em `front`, `back`,
  `not_declared` e `not_applicable`.

#### Arquivos gerados

Todos em `notebooks/colabs_S4/`.

| Arquivo | Formato | Conteúdo |
|---|---|---|
| `dataset_<area>.csv` (7 arquivos) | CSV, *wide* | Uma linha por ensaio. Colunas: `run_idx`; as 7 features categóricas em texto (`blower_direction`, `housing`, `board_position`, `run_case`, `losses`, `deflector_hole`, `material_coil_support`); as 6 colunas `reason_*` e as 4 `loss_profile_order_*` (já 0/1); e uma coluna de temperatura por componente da área, com o nome do termopar. Áreas: `oven_hmi`, `ct_knobs`, `main_vent_sys`, `ipc_cooling`, `mainback`, `ipc_comp`, `coil_temps`. |
| `limites_ativos.csv` | CSV | Um limite por componente. Colunas: `tc_name`, `key_area`, `active_limit` (texto original), `limite_num` (número extraído), `apenas_monitorar` (booleano). |
| `chave_replicata.json` | JSON | `{"chave_por_run_idx": {run_idx: chave}, "n_ensaios": int, "n_configuracoes": int}`. A chave concatena com `\|` as 7 features base, os 8 indicadores candidatos de `derivados_completo`, o texto de `reason` normalizado e o marcador DoE. **Não é variável de entrada em hipótese alguma**, pois é lida apenas pelo cálculo do piso de ruído, no notebook 04. |



---

### 02. `tratamento_datasets_13cat`

**Caminho real:** `notebooks/colabs_S4/tratamento_datasets_13cat.ipynb`

<sub>bibliotecas: pandas, numpy</sub>

#### Objetivo

Transforma os sete conjuntos crus do notebook 01 na **base efetivamente usada pela
modelagem**: remove um sensor defeituoso, duas colunas com excesso de nulos e os componentes
sem nenhuma medição, e aplica one-hot às features mantendo as **13 categorias originais de
`losses`** (daí o nome). É o notebook que produz `dataset_<area>_13cat.csv`, lido por todos os
notebooks de 04 a 07. Nas seções 9 e 10 grava ainda três conjuntos auxiliares.

O próprio markdown registra que existem dois notebooks de tratamento idênticos que diferem
**apenas** no tratamento de `losses` (este e `tratamento_datasets_perfil`), mas na pasta
`colabs_S4` só existe este; a versão `perfil` passou a ser gerada aqui, na seção 10.

#### Arquivos de entrada

| Arquivo | Formato | Origem |
|---|---|---|
| `notebooks/colabs_S4/dataset_<area>.csv` (7) | CSV *wide* | Notebook **01**. Encontrados por `glob("dataset_*.csv")` com filtro que exclui `_13cat` e `_perfil`. O markdown registra que reprocessar as próprias saídas já foi um erro ocorrido. |
| `notebooks/colabs_S4/limites_ativos.csv` | CSV | Notebook **01**. É lido, filtrado e **reescrito no mesmo caminho**. |
| `notebooks/colabs_S3/preparacao_testes/particao_treino_teste.json` | JSON | Externo ao pipeline: gerado por `notebooks/colabs_S3/preparacao_testes/split_treino_teste.ipynb`. Resolvido pela função `localiza`/`localiza_json`, que tenta `.`, `../colabs_S3/preparacao_testes` e `notebooks/colabs_S3/preparacao_testes`. Usado na seção 9 (mediana só do desenvolvimento) e na seção 10 (geração do CSV da partição). |

#### Etapas principais

1. Lê os sete conjuntos do `01`, com um filtro que ignora as próprias saídas.
2. Exclui `Oven RTD Connector Body` do Mainback, cujas leituras ficam sempre acima do limite,
   padrão compatível com defeito de calibração.
3. Exclui `AC Blower Inlet Max` e `AC Blower Inlet Avg` do Main Vent Sys, com 92,9% de nulos.
4. Remove os componentes sem nenhuma medição e os retira também da tabela de limites.
5. Aplica o one-hot às sete features, em valores inteiros, e descarta `run_idx`.
6. Grava, à parte, a versão com IGBTs agrupados e ausentes imputados, a versão com `losses`
   reduzida ao perfil, e a partição em CSV.

**Resultado.** As células de gravação imprimem a forma de cada arquivo e quantas colunas de
`losses` ele tem. A conferência final mostra que cada ensaio soma 0 ou 1 entre as colunas de
`losses`, isto é, que nenhum recebeu duas categorias, e que o one-hot saiu em inteiros.

#### Funções definidas

- `raiz_repositorio(inicio=None)`: idêntica à do `01`.
- `localiza(nome)` e `localiza_json(nome)`: procuram a partição na pasta do notebook ou na
  pasta de preparação da terceira sprint, de modo que o notebook rode no Colab e no
  repositório sem edição.
- `perfil_de_perda(categoria)`: reduz as treze categorias ao perfil de perda, em `simone`,
  `miro` e `no losses`, regra herdada da terceira sprint.

#### Arquivos gerados

Todos em `notebooks/colabs_S4/`.

| Arquivo | Formato | Conteúdo |
|---|---|---|
| `dataset_<area>_13cat.csv` (7) | CSV *wide* | **Base do pipeline.** 56 linhas (ensaios). Colunas, nesta ordem: as dummies 0/1 das 7 features (`blower_direction_*`, `housing_*`, `board_position_*`, `run_case_*`, `losses_*` nas 13 categorias, `deflector_hole_*`, `material_coil_support_*`); as 6 `reason_*` e as 4 `loss_profile_order_*`; e uma coluna de temperatura por componente da área. **Sem `run_idx`**, porque a ordem das linhas *é* o `run_idx`, premissa usada pelos notebooks seguintes. |
| `dataset_<area>_perfil.csv` (7) | CSV *wide* | Igual ao anterior, com as colunas `losses_*` reduzidas aos níveis do perfil de perda, na mesma posição em que estavam as treze. |
| `dataset_<area>_13cat_tratado.csv` (7) | CSV *wide* | Versão com IGBTs em quatro grupos e **nenhum nulo** (mediana do desenvolvimento). Auxiliar. |
| `limites_ativos.csv` | CSV | Mesmo esquema do notebook 01, filtrado: sem os componentes 100% nulos. Reescrito no mesmo caminho. |
| `limites_ativos_tratado.csv` | CSV | Limites com os 16 IGBTs substituídos pelos 4 grupos, cada grupo com o menor limite dos seus membros. Auxiliar. |
| `particao_treino_teste.csv` | CSV | Colunas `run_idx`, `conjunto` (`treino`/`teste`), `dobra` (vazia nos reservados), `run_case`. Cópia em CSV da partição JSON. |



---

### 03. `tratamento_nulos_pos_revisao`

**Caminho real:** `notebooks/colabs_S4/tratamento_nulos_pos_revisao.ipynb`

<sub>bibliotecas: pandas, numpy</sub>

#### Objetivo

Resolve os valores nulos que restam nas **colunas de temperatura** após o notebook 02,
gravando o resultado com o sufixo `_snull`. Decide por coluna entre excluir, imputar a
mediana ou imputar a média.

**Atenção:** apesar de ocupar a posição 03 na ordem de execução, **nenhum dos notebooks 04 a
07 lê os arquivos `_snull`**. Eles trabalham a máscara de ausentes (descartam a leitura sem
medição) em vez de imputá-la. Este notebook é, portanto, um ramo lateral do pipeline.

#### Arquivos de entrada

| Arquivo | Formato | Origem |
|---|---|---|
| `notebooks/colabs_S4/dataset_<area>_13cat.csv` (7) | CSV *wide* | Notebook **02**. Encontrados por `glob(f"dataset_*{versao}.csv")` para cada `versao` em `VERSOES`, excluindo os que já têm `_snull` no nome. |

O cabeçalho do notebook afirma partir de `dataset_*_perfil.csv` **e** `dataset_*_13cat.csv`,
mas a constante é `VERSOES = ("_13cat",)`: na prática só a versão de 13 categorias é
processada. O próprio código registra a restrição e a sua consequência medida.

#### Etapas principais

1. Lê os sete conjuntos `_13cat`.
2. Levanta, para cada coluna com nulos, a proporção de ausentes, o IQR, a faixa sem outliers e
   a contagem de extremos.
3. Decide a ação de cada coluna pelas três regras abaixo e monta a tabela do plano **antes**
   de alterar qualquer dado, de modo que a decisão possa ser conferida.
4. Aplica o plano sobre cópias dos conjuntos.
5. Confere, por `assert`, que não restou nulo, que o número de linhas não mudou e que apenas
   as colunas previstas saíram.
6. Grava os arquivos `_snull` e imprime o resumo das mudanças.

**Resultado.** O plano sai como tabela, com a ação e o motivo de cada coluna, e o resumo
final compara, por conjunto, o número de colunas e de nulos antes e depois. A conferência
cruzada entre as duas versões da base é declarada inaplicável, porque esta sprint processa
apenas a codificação de treze categorias.

#### Regras aplicadas, nesta ordem

| # | Condição | Ação |
|---|---|---|
| 1 | mais de `PCT_NULOS_MAX` (20%) de nulos **e** faixa de temperatura > `FAIXA_MAX` (35 °C) | **excluir** a coluna |
| 2 | possui outlier além de `MULT_EXTREMO` (2,5) vezes o IQR | preencher com a **mediana** |
| 3 | zero outliers extremos | preencher com a **média** |

Duas decisões de implementação estão documentadas no notebook. A **faixa é medida sem os
outliers**, isto é, o intervalo dentro da cerca de 1,5×IQR (os bigodes do boxplot), e não a
amplitude bruta, para que um único ponto distante não cause a exclusão. E a escolha entre
média e mediana **segue a assimetria**: a média é sensível a extremos, a mediana não.

#### Funções definidas

- `perfila_coluna(serie)`: reúne num dicionário tudo que as regras precisam, para que os
  quartis não sejam recalculados a cada regra.
- `colunas_com_nulos(df)`: lista as colunas numéricas que têm ao menos um ausente.
- `decide_acao(perfil)`: traduz as três regras em uma ação e o seu motivo. Separar a decisão
  da execução é o que permite conferir o plano inteiro antes de tocar no dado.
- `raiz_repositorio(inicio=None)`: idêntica à do `01`.

#### Arquivos gerados

| Arquivo | Formato | Conteúdo |
|---|---|---|
| `notebooks/colabs_S4/dataset_<area>_13cat_snull.csv` (7) | CSV *wide* | Mesmo esquema de `dataset_<area>_13cat.csv`, **sem nenhum nulo**: colunas excluídas pela regra 1 não aparecem; as demais têm os ausentes preenchidos por média ou mediana da própria coluna. Mesmo número de linhas da entrada (garantido por `assert`). |



---

### 04. `modelagem`

**Caminho real:** `notebooks/colabs_S4/modelagem.ipynb`

<sub>bibliotecas: pandas, numpy, scipy, scikit-learn (ensemble, linear_model, neural_network, pipeline, preprocessing, compose), joblib, lazypredict (opcional, atrás de EXECUTAR_VARREDURA = False)</sub>

#### Objetivo

Notebook central do projeto, com **24 seções**. Define as métricas canônicas, mede o piso de
ruído da base, avalia os estimadores um a um (do Ridge ao Gradient Boosting), registra as
alternativas descartadas, e fixa na **seção 23 a formulação entregue da Sprint 4**. É a
seção 23 que `src/frontend/dados/exportar.py` importa por `exec` para treinar os modelos do
dashboard, ou seja, este é o notebook que define o modelo final.

O mapa do próprio notebook distribui as seções assim:

| Seções | Conteúdo |
|---|---|
| 1 a 6 | Configuração, leitura, partição fixa e definição das métricas |
| 7 a 14 | Modelos avaliados um a um e a decisão contra o limite de segurança |
| 15 a 18 | Alternativas descartadas, valores extremos, comparação entre versões da base e as placas de alumínio |
| 19 e 20 | Formulação de referência (Random Forest em todas as regiões), na validação e no teste reservado |
| 21 e 22 | Seleção por região de codificação, estimador e escala do alvo; variabilidade da estimativa |
| **23** | **Formulação entregue da quarta sprint** |
| 24 | Diagnóstico do R² baixo do Oven HMI |

#### Arquivos de entrada

| Arquivo | Formato | Origem |
|---|---|---|
| `notebooks/colabs_S4/dataset_<area>_13cat.csv` (7) | CSV *wide* | Notebook **02**. `SUFIXO = "_13cat"`, `PASTA_DADOS = "colabs_S4"`. |
| `notebooks/colabs_S4/dataset_<area>_perfil.csv` (7) | CSV *wide* | Notebook **02**, seção 10. Lido pelas seções 17, 21 e 22, cujo objeto é comparar as duas codificações. `PASTA_DADOS_PERFIL = "colabs_S4"`. |
| `notebooks/colabs_S4/limites_ativos.csv` | CSV | Notebooks **01**/**02**. |
| `notebooks/colabs_S3/preparacao_testes/particao_treino_teste.json` | JSON | Externo: `split_treino_teste.ipynb`. `PASTA_PARTICAO = "colabs_S3/preparacao_testes"`. |
| `notebooks/colabs_S4/particao_treino_teste.csv` | CSV | Notebook **02**, seção 10. Lido pela seção 24. |
| `notebooks/colabs_S4/chave_replicata.json` | JSON | Notebook **01**. Lido pela seção 7, para o piso de ruído. |

**Não lê** os arquivos `_snull` nem os `_tratado`.

#### Etapas principais

O notebook tem vinte e quatro seções. Em blocos:

1. **Seções 1 a 6:** leitura dos conjuntos, da tabela de limites e da partição fixa, e a
   definição canônica das métricas.
2. **Seções 7 a 14:** os estimadores avaliados um a um, do modelo linear de referência ao
   Gradient Boosting, o piso de ruído da base e a decisão contra o limite de segurança.
3. **Seções 15 a 18:** as alternativas descartadas, a investigação dos valores extremos, a
   comparação entre as duas codificações e o caso das placas de alumínio.
4. **Seções 19 e 20:** a formulação de referência, com Random Forest em todas as regiões, na
   validação e no conjunto de teste reservado.
5. **Seções 21 e 22:** a seleção de codificação, estimador e escala do alvo por região, e a
   variabilidade da estimativa.
6. **Seção 23:** a formulação entregue, descrita abaixo.
7. **Seção 24:** o diagnóstico do coeficiente de determinação do Oven HMI.

**Resultado.** A formulação entregue mede de 5,96 a 6,23 °C de erro médio absoluto na
validação, conforme o método de aninhamento, contra 6,47 °C da formulação de
referência, com Random Forest em todas as áreas (seção 19 de `modelagem.ipynb`), e 6,565 °C
do modelo da Sprint 3, este medido na base anterior. Três resultados negativos ficam registrados e devem
ser lidos antes de qualquer proposta de ampliar a busca: a seleção sobre doze combinações
rende menos do que sobre duas; as colunas de ordem de chaveamento pioram o resultado; e o
conjunto de teste reservado aponta em sentido contrário ao da validação.

#### A formulação entregue (seção 23)

A regra está escrita como constante `_REGRA`, e não lida da seleção aninhada, "para que a
entrega não dependa da ordem de execução das células"; a célula seguinte confere se as duas
coincidem e **relata** divergência em vez de abortar.

| Área | Estimador | Tratamento de `losses` |
|---|---|---|
| `oven_hmi` | `rf` (Random Forest) | agrupado |
| `ct_knobs` | `rf` | detalhado |
| `main_vent_sys` | `et+gb` (média Extra Trees + Gradient Boosting) | detalhado |
| `ipc_cooling` | `rf` | agrupado |
| `mainback` | `rf` | detalhado |
| `ipc_comp` | `rf` | agrupado |
| `coil_temps` | `rf+gb` (média Random Forest + Gradient Boosting) | detalhado |

Todos com `n_estimators=300` e os demais hiperparâmetros no padrão da biblioteca, mais o
**peso próximo ao limite** no ajuste. `_SEMENTE_ENTREGA = 42`.

Resultados que o notebook reporta para ela: **5,96 a 6,23 graus** de erro na validação
cruzada, conforme o método de aninhamento, contra 6,107 da formulação que escolhia apenas o
estimador, 6,47 da formulação de referência (seção 19 de `modelagem.ipynb`) e 6,565 do modelo
da Sprint 3. No conjunto de teste reservado mede **5,154 graus**, contra 5,17 da formulação que
escolhia apenas o estimador, 5,15 da formulação de referência (seção 20) e 4,37 do modelo da
Sprint 3, este medido na base anterior e por isso não diretamente comparável.

#### Funções definidas

São cerca de vinte, agrupadas aqui pelo papel. As de nome iniciado por sublinhado são
auxiliares de uma seção específica.

**Infraestrutura**

- `raiz_repositorio(inicio=None)` e `resolve(pasta, arquivo)`: localizam os arquivos, com
  prioridade para a pasta declarada, de modo que um homônimo vizinho não seja lido no lugar.
- `limite_de(componente)`: limite de segurança tabelado, ou `None`.

**Preparo dos dados**

- `separar(df, prefixos)`: divide as colunas entre entradas e alvos pelo prefixo.
- `indices_da_dobra(df, dobra, y_col)`: índices de treino e validação de uma dobra, já com a
  máscara de ausentes do alvo.
- `formato_longo(nome, df)` e `formato_longo_completo(df)`: montam uma linha por par de ensaio
  e componente, o segundo conservando os ensaios reservados.
- `atributos_do_componente(longo, treino)` e `matriz(...)`: calculam o patamar e a escala de
  cada componente **apenas no treino da dobra** e os acrescentam às entradas, o que impede o
  vazamento do alvo por construção (KAUFMAN et al., 2012).

**Métricas, definição canônica do projeto**

- `_valida(y_real, y_previsto)`: recusa entradas que produziriam métrica sem sentido, entre
  elas alvo não positivo, que denunciaria valores padronizados no lugar de graus Celsius.
- `avaliar(y_real, y_previsto, limite=None)`: erro absoluto médio, raiz do erro quadrático,
  coeficiente de determinação, viés e erro percentual; com limite informado, acrescenta as
  métricas de decisão, entre elas os falsos aprovados.
- `consolidar(tabela, grupo)`: agrega por área em macro-média, que pesa igual cada componente,
  e micro-média, que pesa pelo número de observações.

**Diagnóstico**

- `piso_de_ruido(df)`: estima o erro irredutível pela discordância entre ensaios de
  configuração idêntica, com a chave lida de arquivo e não derivada do vetor de entrada.
- `decisao_com_margem(tabela, predicoes, margem)`: recalcula a triagem somando a margem à
  previsão.

**Formulação entregue, seção 23**

- `_familia_de_perdas(coluna)` e `_bloco_agrupado(d, colunas)`: reduzem os treze indicadores de
  `losses` a níveis exclusivos, com verificação de exclusividade.
- `_entradas`, `_longo` e `_carrega(area, tratamento)`: montam o formato longo da região, sem
  o conjunto de teste reservado.
- `_peso_perto_do_limite(y, comps)`: dá peso três às leituras sobre o limite e praticamente um
  às distantes trinta graus dele, porque é junto ao limite que a triagem decide.
- `_ajusta(Xt, yt, Xv, estimador, semente, peso)`: ajusta o Random Forest, o Extra Trees ou a média
  de um deles com o Gradient Boosting, todos com trezentas árvores.
- `_fora_da_dobra(area, candidato, semente)`: erro de cada leitura quando a sua configuração
  está fora do treino.

##### Funções importadas de módulos `.py`

Nenhuma. O notebook não importa módulo próprio do repositório. Em sentido inverso,
`src/frontend/dados/exportar.py` lê este notebook e executa o código da seção 23 por `exec`, em
vez de copiá-lo, de modo que uma mudança ali chega ao dashboard na exportação seguinte.

#### Arquivos gerados

**Nenhum.** Não há `to_csv`, `json.dump`, `savefig` nem `joblib.dump` em nenhuma célula. As
saídas são tabelas, textos e gráficos renderizados nas próprias células. A seção 23 é
"materializada" em arquivo depois, por `src/frontend/dados/exportar.py`.



---

### 05. `random_search_13cat`

**Caminho real:** `notebooks/colabs_S4/random_search_13cat.ipynb`

<sub>bibliotecas: pandas, numpy, scipy, scikit-learn (base, ensemble, linear_model, metrics, model_selection, pipeline, preprocessing)</sub>

#### Objetivo

Busca aleatória de hiperparâmetros para **Ridge, Random Forest, Extra Trees e Gradient
Boosting** em cada uma das sete áreas, sobre a versão 13cat da base, mantendo a formulação da
solução entregue na Sprint 3: um modelo por área, no formato longo, com a identidade do
componente substituída por patamar e escala calculados só no treino de cada dobra. Existe
neste ponto para responder se os hiperparâmetros fixos da modelagem podem ser melhorados,
sem mudar a formulação.

**Não é** o mesmo arquivo que `notebooks/colabs_S4/random_search.ipynb` (94 MB, com saídas
gravadas), que é uma execução anterior sobre a base `perfil` e não faz parte desta ordem.

#### Arquivos de entrada

Todos lidos **apenas** de `notebooks/colabs_S4/` (`PASTA_DADOS = "."`), pela função `caminho`.

| Arquivo | Formato | Origem |
|---|---|---|
| `notebooks/colabs_S4/dataset_<area>_13cat.csv` (7) | CSV *wide* | Notebook **02**. `VERSAO = "13cat"`. |
| `notebooks/colabs_S4/particao_treino_teste.csv` | CSV | Notebook **02**, seção 10. |
| `notebooks/colabs_S4/limites_ativos.csv` | CSV | Notebooks **01**/**02**. |

Com `RETOMAR = True`, lê também os seus próprios `resultados_random_search/kpis_<area>_<alg>.csv`
de uma execução anterior, para continuar do ponto em que parou.

#### Etapas principais

1. Lê os sete conjuntos, a tabela de limites e a partição, todos da própria pasta.
2. Monta o formato longo de cada área, **apenas com os ensaios de treino**, descartando as
   leituras sem temperatura por máscara.
3. Define o transformador que substitui a identidade do componente por patamar e escala, e os
   pipelines de cada algoritmo.
4. Sorteia os candidatos por busca aleatória (BERGSTRA; BENGIO, 2012) e os avalia por validação
   que retira uma configuração inteira por vez.
5. Reavalia a configuração da terceira sprint e os três melhores candidatos com os indicadores
   completos, inclusive os falsos aprovados em três margens.
6. Consolida por área e por algoritmo e grava os resultados intermediários.

**Resultado.** Com `reason` entre as entradas, o Gradient Boosting buscado tem o menor erro
nas sete áreas e mede 5,87 °C somadas, com 31 aprovações indevidas na margem de oito graus.
Esse valor é otimista, porque a mesma validação que escolheu o candidato também o avalia
(CAWLEY; TALBOT, 2010); o notebook registra que o ganho só deve ser tratado como real quando
superar o limiar de ruído de 0,5 °C adotado pelo projeto.

#### Funções e classes definidas

**Leitura**

- `caminho(arquivo)`: monta o caminho na pasta do notebook e falha cedo se o arquivo faltar.
- `carrega_longo(area)`: monta o formato longo e interrompe a execução se aparecer coluna que
  não seja entrada nem alvo, para que uma mudança de nome nunca descarte variáveis em silêncio.

**Representação do componente**

- `EstatisticasComponente`: transformador compatível com o scikit-learn (PEDREGOSA et al.,
  2011) que troca a identidade do componente pelo patamar e pela escala, ajustados **só no
  treino de cada dobra**.
- `cria_pipeline(algoritmo)`: encadeia o transformador, o escalonador, aplicado apenas ao
  modelo linear, e o estimador.

**Avaliação e busca**

- `kpis(longo, previsto)`: indicadores sobre as previsões acumuladas das dezessete dobras.
- `avalia(algoritmo, params, ...)`: acumula as previsões fora da dobra e devolve os
  indicadores.
- `busca(area, algoritmo, ...)`: executa a busca aleatória, grava a tabela de candidatos e
  reavalia os melhores.
- `agrega(t)`: consolida as sete áreas por algoritmo, ponderando pelo número de previsões.

##### Funções importadas de módulos `.py`

Nenhuma. Existe em `notebooks/colabs_S4/artefatos/hiperparametros.py` um módulo que lê os
hiperparâmetros ajustados, mas a sua origem declarada é outro notebook, e nenhum dos sete
deste pipeline o importa.

#### Arquivos gerados

Todos em `notebooks/colabs_S4/resultados_random_search/` (`PASTA_SAIDA`), **pasta não
versionada**; o próprio cabeçalho pede que seja apagada sempre que a base ou o espaço de
busca mudarem.

| Arquivo | Formato | Conteúdo |
|---|---|---|
| `cv_<area>_<algoritmo>.csv` (28) | CSV | `cv_results_` do `RandomizedSearchCV`, sem as colunas `split*`, ordenado por `rank_test_score`, com `MAE_val_dobras` e `MAE_treino` acrescentadas. |
| `kpis_<area>_<algoritmo>.csv` (28) | CSV | Uma linha por candidato reavaliado (`sprint3` mais `rs_top1..rs_top3`). Colunas: `area`, `algoritmo`, `candidato`, os KPIs de `kpis`, `tempo_s`, `params` (JSON) e `tempo_busca_s`. |
| `kpis_consolidado.csv` | CSV | Concatenação de todos os `kpis_<area>_<alg>.csv`. |

**Nenhum modelo treinado é gravado**, porque `refit=False`, e só as métricas e os hiperparâmetros
em texto saem em arquivo. O notebook 06 registra justamente isso: "aquele caderno grava os
hiperparâmetros só em arquivos fora do repositório" (o termo é dele), motivo pelo qual os valores de IPC
Cooling tiveram de ser obtidos refazendo a busca com a mesma semente.



---

### 06. `comparacao_modelos_final`

**Caminho real:** `notebooks/colabs_S4/comparacao_modelos_final.ipynb`

<sub>bibliotecas: pandas, numpy, scipy, scikit-learn (base, ensemble, model_selection, pipeline), matplotlib, shap (instalada por pip na própria célula)</sub>

> Este notebook se declara a segunda versão da comparação: acrescenta o Extra Trees e traz o
> quadro consolidado da seção 11. A primeira versão,
> `notebooks/colabs_S4/comparacao_modelos.ipynb`, permanece intacta como registro da comparação
> original entre Random Forest, Gradient Boosting e a mescla dos dois.

#### Objetivo

Compara, sob o mesmo protocolo e o mesmo espaço de busca, **Random Forest, Gradient Boosting,
Extra Trees e a mescla** Random Forest+Gradient Boosting, em cada área; confronta a configuração fixa com
a buscada e mantém a melhor; explica o modelo com valores SHAP; reproduz a formulação
entregue para comparação direta; e monta o **quadro consolidado** com o melhor resultado
conhecido de cada área térmica entre todos os notebooks de `colabs_S4`.

Três mudanças em relação à v1, conforme o próprio cabeçalho: o Extra Trees entra na
comparação; a seção 11 reúne o melhor de cada área com o tratamento de dados que ele usa; e a
seção 7.1 mostra, no Oven HMI, quanto as métricas melhorariam se a validação fosse por linha
em vez de por configuração, e por que esse corte não foi adotado.

#### Arquivos de entrada

| Arquivo | Formato | Origem |
|---|---|---|
| `notebooks/colabs_S4/dataset_<area>_13cat.csv` (7) | CSV *wide* | Notebook **02**. |
| `notebooks/colabs_S4/limites_ativos.csv` | CSV | Notebooks **01**/**02**. |
| `notebooks/colabs_S3/preparacao_testes/particao_treino_teste.json` | JSON | Externo: `split_treino_teste.ipynb`. |

A função `localiza` restringe os CSVs a `.` e `notebooks/colabs_S4`; **só a partição JSON**
pode vir de `../colabs_S3/preparacao_testes` ou `notebooks/colabs_S3/preparacao_testes`. No
Colab, a primeira célula de código pede o envio dos nove arquivos. Os dez ensaios do conjunto
de teste reservado **não são lidos** neste notebook.

#### Etapas principais

1. Lê os sete conjuntos, a tabela de limites e a partição; os ensaios reservados não são
   abertos.
2. Monta o formato longo de cada área, com verificação de que toda coluna foi classificada.
3. Define as métricas a partir do uso pretendido, que é a triagem prévia de componentes.
4. Avalia Random Forest (BREIMAN, 2001), Gradient Boosting (FRIEDMAN, 2001), Extra Trees
   (GEURTS; ERNST; WEHENKEL, 2006) e a média entre os dois primeiros.
5. Compara, para cada algoritmo, a configuração fixa com a melhor de quinze sorteadas, e
   mantém a melhor das duas.
6. Explica o modelo de IPC Comp por valores de Shapley (LUNDBERG; LEE, 2017), agrupando as
   contribuições por família de variável.
7. Reproduz a formulação entregue e monta o quadro consolidado do melhor resultado de cada
   área.

**Resultado.** A formulação entregue mede 5,855 °C, com viés de +0,152, 23 aprovações
indevidas e detecção de 88,1%, à frente do melhor modelo por área deste notebook, que mede
5,950 °C mesmo com busca de hiperparâmetros. A explicação por Shapley atribui cerca de um
terço ao patamar do componente e cerca de 40% ao tipo de ensaio somado a `losses`, o
que é coerente com a física da área; nenhuma variável implausível aparece entre as de maior
peso.

#### Funções e classes definidas

**Leitura e métricas**

- `localiza(nome)` e `carrega(area)`: resolvem os arquivos e montam o formato longo apenas com
  os ensaios de desenvolvimento.
- `metricas(y, pred, componentes)`: erro, viés e os três indicadores de triagem sob a margem
  de oito graus, que são as aprovações indevidas, as reprovações desnecessárias e a detecção.

**Modelos**

- `EstatisticasComponente` e `pipeline(estimador)`: a mesma representação do componente usada
  no `05`, aqui por cópia e não por importação.
- `floresta(**p)`, `gradiente(**p)` e `extra(**p)`: fábricas dos três estimadores, com a
  semente fixa.
- `divisoes(area, X, y, grupos)` e `prever(fabrica, X, y, divs)`: retiram uma configuração
  inteira por vez e devolvem a previsão de cada leitura quando ela está fora do treino.

**Formulação entregue**

- `agrupa_perdas(X)`: substitui os treze indicadores por níveis exclusivos, com verificação de
  exclusividade.
- `peso_limite(y, componentes)`: o mesmo peso junto ao limite da seção 23 do `04`.
- `prever_entrega(area)`: aplica a regra por área, com o agrupamento de `losses` e o peso, e
  usa Extra Trees no lugar do Random Forest em Main Vent Sys.

**Apoio**

- `familia(coluna)`: agrupa as colunas por família, para que a explicação seja lida em termos
  do problema e não de colunas isoladas.

##### Funções importadas de módulos `.py`

Nenhuma.

#### Arquivos gerados

**Nenhum.** Não há `to_csv`, `json.dump` nem `savefig`. Os três gráficos (as barras de
contribuição por família, o beeswarm SHAP e o painel da terceira para a quarta sprint)
são exibidos por `plt.show()` e **não são salvos em arquivo**. A seção 11 e a seção 12
imprimem as tabelas também em markdown, para serem copiadas na documentação.



---

### 07. `analises_complementares`

**Caminho real:** `notebooks/colabs_S4/analises_complementares.ipynb`

<sub>bibliotecas: pandas, numpy, scikit-learn (ensemble), joblib, flaml (instalada por pip na própria célula), pycaret (opcional, exige Python 3.11 ou anterior)</sub>

#### Objetivo

Reúne as cinco análises de fechamento da sprint, que sustentam decisões registradas na
documentação mas não cabiam nos notebooks de modelagem e de comparação. É o último notebook da
ordem porque todas as cinco pressupõem a formulação entregue já fixada.

1. **Grade ampla por área**: vinte candidatos por área, com a escolha medida pelo
   aninhamento estrito; sustenta a troca do estimador em Main Vent Sys e a recusa da mesma
   troca em Coil Temps.
2. **Bootstrap** sobre a formulação entregue, que dá o intervalo das métricas e uma segunda
   estimativa do erro.
3. **AutoML com FLAML**, com a escolha de algoritmo e hiperparâmetros refeita em cada dobra.
4. **AutoML com PyCaret**, que exige Python 3.11 e por isso roda em ambiente separado.
5. **Oven HMI**: o efeito de contar duas vezes o mesmo sensor, que explica por que
   resultados obtidos na base antiga pareciam melhores.

Todas seguem o protocolo do projeto: 17 dobras por configuração, patamar e escala calculados
só no treino de cada dobra, e o peso próximo ao limite. O conjunto de teste reservado **não é
consultado**.

#### Arquivos de entrada

| Arquivo | Formato | Origem |
|---|---|---|
| `notebooks/colabs_S4/dataset_<area>_13cat.csv` (7) | CSV *wide* | Notebook **02**. |
| `notebooks/colabs_S4/limites_ativos.csv` | CSV | Notebooks **01**/**02**. |
| `notebooks/colabs_S4/particao_treino_teste.csv` | CSV | Notebook **02**, seção 10. |

Lidos **por nome puro**, sem função de resolução: o notebook depende do `chdir` da primeira
célula e tem `assert all(os.path.exists(f) for f in ARQUIVOS), "execute a partir da pasta
colabs_S4"`. No Colab, pede o envio dos nove arquivos.

#### Etapas principais

1. Lê os sete conjuntos, a tabela de limites e a partição, e define o peso junto ao limite e as
   métricas comuns às cinco análises.
2. **Grade ampla:** vinte candidatos por área, com a escolha medida por aninhamento estrito,
   em que cada dobra externa escolhe usando apenas as demais configurações.
3. **Bootstrap:** cinco mil reamostras das dezessete configurações sobre as previsões já
   calculadas, e cinquenta repetições de treino em amostra com reposição.
4. **AutoML com FLAML:** a escolha de algoritmo e hiperparâmetros refeita em cada dobra, com
   orçamento de sessenta segundos.
5. **AutoML com PyCaret:** cerca de vinte regressores comparados por dobra, em ambiente
   separado, por exigir Python 3.11 ou anterior.
6. **Oven HMI:** o efeito de contar duas vezes o mesmo sensor.

**Resultado.** A grade ampla sustenta a troca do estimador em Main Vent Sys e recusa a mesma
troca em Coil Temps, onde a mescla reduz o erro médio mas dobra as aprovações indevidas. As
duas ferramentas de AutoML não superam a formulação entregue: no agregado, o PyCaret mede
6,267 °C contra 5,853 °C da entrega. A última análise explica por que resultados obtidos na
base antiga pareciam melhores: o componente duplicado entrava duas vezes na média.

#### Funções definidas

**Montagem dos dados**

- `entradas(area, trat, motivo)`: monta o formato longo da área, com `losses` no tratamento detalhado ou
  agrupado e com `reason` presente ou ausente, sem o conjunto reservado.
- `familia(c)`: nível do perfil de perda a partir do nome da coluna.
- `matriz(X, y, c, tr, alvo)`: acrescenta o patamar e a escala calculados apenas nas linhas de
  treino.

**Ajuste e medição**

- `peso(y, comps)`: o mesmo peso junto ao limite da seção 23 do `04`.
- `ajusta_variante(...)`: ajusta Random Forest, Extra Trees e Gradient Boosting e devolve também as duas
  médias entre eles.
- `metricas(y, p, co)`: erro, viés e os indicadores de triagem, ignorando previsões ausentes.

**Condução das análises**

- `tarefa(area, semente, dobra)`: executa uma dobra externa da grade ampla e acumula o erro
  interno de cada candidato.
- `formula(area, semente, modo)`: aplica um dos modos de escolha, entre regra fixa e
  aninhamento estrito.
- `metricas_b(idx)` e `repeticao(b)`: conduzem as duas aplicações do bootstrap.
- `dobra_flaml(area, k)` e `dobra_pycaret(area, k)`: entregam o treino externo a cada
  ferramenta de AutoML e preveem a dobra retida, sem reajustar o modelo com ela.

##### Funções importadas de módulos `.py`

Nenhuma.

#### Arquivos gerados

**Nenhum.** Não há `to_csv`, `json.dump`, `savefig` nem `joblib.dump`. Todas as saídas são
tabelas impressas nas células.



---

### Aplicação: `exportar.py`

**Caminho real:** `src/frontend/dados/exportar.py`

<sub>bibliotecas: pandas, numpy, scikit-learn (ensemble, preprocessing), joblib</sub>

#### Objetivo

Transformar a formulação escolhida na seção 23 do notebook `04` nos arquivos que o dashboard
consulta. **Não é notebook e não faz parte da pasta `notebooks/`**, mas é a última etapa do
pipeline: sem ele, o modelo final existe como decisão registrada e não como artefato.

Está documentado aqui porque nenhum dos sete notebooks grava modelo treinado. Os notebooks
`04` a `07` produzem apenas tabelas e gráficos nas próprias células.

A escolha de projeto que sustenta o script é ler o código da seção 23 **por `exec`, em vez de
copiá-lo**, de modo que uma mudança naquela seção chegue ao dashboard na exportação seguinte,
sem que haja duas versões da mesma receita em lugares diferentes.

#### Arquivos de entrada

| Arquivo | Formato | Origem |
|---|---|---|
| `notebooks/colabs_S4/dataset_<area>_13cat.csv` (7) | CSV *wide* | Notebook **02** |
| `notebooks/colabs_S4/dataset_<area>.csv` (7) | CSV *wide* | Notebook **01**, para os rótulos originais de cada ensaio |
| `notebooks/colabs_S4/limites_ativos.csv` | CSV | Notebooks **01** e **02** |
| `notebooks/colabs_S3/preparacao_testes/particao_treino_teste.json` | JSON | Externa ao pipeline |
| `notebooks/colabs_S4/modelagem.ipynb` | Notebook | A célula da seção 23 é executada por `exec`; da célula da regra, apenas a constante `_REGRA` é extraída, **sem execução**, porque aquela célula também avalia o conjunto reservado |
| `notebooks/colabs_S3/preparacao_testes/metricas.ipynb` | Notebook | A célula que define `avaliar` e `consolidar`, igualmente por `exec` |

As pastas podem ser trocadas pelas variáveis de ambiente `IGNIS_PASTA_NB`,
`IGNIS_PASTA_PARTICAO` e `IGNIS_SAIDA`, o que permite exportar a partir dos dados de outro
teste sem editar o código.

#### Etapas principais

1. Carrega a função canônica de métricas e, da seção 23, as funções do modelo e a regra por
   área, localizando as células **pelo conteúdo e não pela posição**.
2. Reproduz a validação cruzada das sete áreas, com dezessete dobras, usando
   `_fora_da_dobra` da própria seção 23.
3. **Confere o resultado** contra o erro de cada candidato registrado na seção 23, com
   tolerância de 0,10 °C, e interrompe se divergir.
4. Treina o modelo final de cada área sobre **todos os ensaios de desenvolvimento**, aplica
   `StandardScaler` e grava o pacote em `modelos/modelos.joblib`.
5. Calcula as métricas por componente e por área e decide a **viabilidade** de cada área.
6. Exporta as predições de validação, as importâncias das variáveis, os ensaios, o catálogo
   de componentes, os parâmetros de leitura e a geometria 3D.

**Resultado.** O script imprime, por área, o erro e a proporção de predições dentro da margem
de 10%, e confirma na saída que a reprodução bate com a seção 23. Cada área recebe um veredito
de viabilidade pela regra registrada no próprio arquivo: é viável quando ao menos **70%** das
predições caem dentro da margem e os falsos aprovados não passam de **5%** das predições.

#### Funções definidas

**Leitura do que está nos notebooks**

- `carregar_funcao_canonica()`: executa a célula do `metricas.ipynb` que define `avaliar` e
  `consolidar`, de modo que o dashboard use exatamente as métricas do projeto.
- `carregar_modelo_s4()`: extrai da seção 23 as funções do modelo e a constante `_REGRA`. A
  leitura para antes da linha que inicia o treino de três sementes, que não é executado.

**Apoio**

- `slug(nome)`: converte o nome do componente em identificador sem acento, usado nos arquivos
  de saída.
- `limpo(v)` e `inteiro(v)`: convertem valores de `numpy` e `NaN` para algo que o JSON aceite.
- `registrar_teste(metricas, particao)`: mantém `testes.json`, a lista de conjuntos de dados
  que o dashboard oferece.
- `main()`: conduz as oito etapas acima e devolve o código de saída.

#### Arquivos gerados

Todos em `src/frontend/dados/`.

| Arquivo | Formato | Conteúdo |
|---|---|---|
| `modelos/modelos.joblib` | joblib | **O modelo final.** Um pacote por área, com o estimador ajustado, o escalonador, o patamar e a escala de cada componente e a lista de colunas de entrada. Recebe uma versão derivada de um resumo da regra, da semente e do número de leituras, gravada junto a cada configuração salva no histórico para acusar previsões feitas com outro modelo. **Fora do git**, por conter árvores ajustadas sobre as temperaturas da parceira |
| `componentes.csv` | CSV | Catálogo: limite, se é apenas monitorado, número de leituras, veredito e o código mascarado usado pelo histórico |
| `metricas_areas.json` | JSON | Métricas por área, o critério de viabilidade e as métricas por dobra |
| `metricas_componentes.json` | JSON | Métricas por componente, com falsos aprovados e reprovados |
| `predicoes.json` | JSON | Predições de validação: real, previsto, dobra, ensaio e componente |
| `importancias.json` | JSON | Importância das variáveis por área; nas áreas de mescla, a média das importâncias dos dois estimadores |
| `ensaios.json` | JSON | Uma entrada por ensaio e área, usada nas coordenadas paralelas e na comparação |
| `geometria_3d.json` | JSON | Regiões, vetores de explosão e posições dos componentes, em milímetros |
| `parametros.json` | JSON | Margem do TAPI, alvo de viabilidade, folga de atenção e versão do modelo |

Quem consome esses arquivos é `src/frontend/previsao.py`, na rota `/api/prever` do servidor.

#### Diferenças em relação aos números oficiais

Três, todas declaradas no próprio script e que explicam pequenas divergências:

- a exportação usa **uma semente (42)**, ao passo que os valores oficiais da documentação são
  média de três, com a escolha refeita dentro das dobras;
- o modelo final é ajustado sobre **todos os ensaios de desenvolvimento**, e não dobra a
  dobra, o que vale para a importância das variáveis e para a previsão sob demanda;
- a conferência contra os valores de referência só é aplicada ao conjunto original da
  parceria, e é desligada por `IGNIS_CONFERIR=0` ao exportar outro teste, que tem números
  próprios e não pode ser comparado com aqueles.

Em nenhum momento o conjunto de teste reservado é lido.


---

## Saídas sem consumidor

Arquivos que o pipeline grava e que nenhuma etapa seguinte lê. Valem como registro: quem for
mexer no pipeline precisa saber o que pode ser removido sem quebrar nada, e o que só existe
por causa de experimentos laterais.

| Arquivo | Produzido por | Quem lê |
|---|---|---|
| `dataset_<area>_13cat_tratado.csv` | 02, seção 9 | **Nenhum notebook de 03 a 07.** Só `testes_extras/testes_treinamento/treino_gradient_boosting.ipynb`. |
| `limites_ativos_tratado.csv` | 02, seção 9 | Idem. |
| `resultados_random_search/*.csv` | 05 | **Nenhum notebook.** São consultados a olho; o notebook 06 registra que precisou **refazer a busca** para recuperar hiperparâmetros que só existem nesses arquivos fora do repositório. |

As entradas e saídas de cada etapa estão na seção [Notebooks](#notebooks), em **Arquivos de entrada** e **Arquivos gerados**.


## Arquivos externos ao pipeline que são necessários

Quatro arquivos que o pipeline consome e não produz. Os dois primeiros são indispensáveis:
sem eles, nada roda.

- **`documents/outros/dataset_original.xlsx`** é o dado bruto, única fonte primária de todo o
  projeto, e sem ele o notebook `01` não roda. Veio da parceira. **Não é versionado**: `*.xlsx` consta do
  `.gitignore`, e `git ls-files` não lista nenhum. Existe apenas na cópia local de quem já o
  recebeu, de modo que não se pode contar com ele num clone novo.

- **`notebooks/colabs_S3/preparacao_testes/particao_treino_teste.json`** é a âncora do
  projeto: define os 10 ensaios reservados, os 46 de desenvolvimento e as 17 dobras. Todo
  número medido desde a terceira sprint pressupõe exatamente esta divisão. Está versionado, e
  por ser JSON escapa do `.gitignore`.

  Foi gerado por `split_treino_teste.ipynb`, que **não faz parte desta ordem** e não deve ser
  reexecutado. Aquele notebook tem uma trava explícita: se a partição recalculada divergir da
  gravada, ele levanta `AssertionError` e não escreve nada. A chave é a configuração sob o
  perfil de `losses`, com `SEMENTE = 42` e `FRACAO_TESTE = 0.20`.

- **`notebooks/colabs_S3/preparacao_testes/metricas.ipynb`** guarda a definição canônica das
  métricas, com a fórmula, a justificativa de cada escolha e a bateria de casos de teste que as
  verifica. A seção 6 do notebook `04` declara copiá-la sem alteração, e o `exportar.py` a
  executa por `exec`. Versionado.

- **`notebooks/colabs_S4/chave_replicata.json`** traz a descrição mais fina disponível de cada
  ensaio, usada apenas para estimar o piso de ruído da base. É gerado pelo notebook `01` e,
  por ser JSON, fica versionado.

### Módulos auxiliares `.py`

**Nenhum dos sete notebooks do pipeline importa módulo `.py` do repositório.** Cada um
redefine as suas próprias funções, inclusive as que se repetem: `raiz_repositorio` aparece nos
sete, `EstatisticasComponente` no `05` e no `06`, e o peso junto ao limite no `04`, no `06` e
no `07`.

Existem módulos relacionados, todos **fora** do pipeline:

- **`src/frontend/dados/exportar.py`** reproduz a formulação entregue e grava os artefatos do
  dashboard. É executado à mão, com `python src/frontend/dados/exportar.py`.

- **`src/frontend/previsao.py`** faz a previsão sob demanda a partir do `modelos.joblib`, e é
  chamado por `src/frontend/servir.py` na rota `/api/prever`.

- **`src/frontend/dados/mascara.py`** gera o código mascarado dos componentes e é importado pelo
  `exportar.py`.

- **`notebooks/colabs_S4/artefatos/hiperparametros.py`** lê o `hiperparametros.json` e
  instancia o estimador de um par de área e algoritmo, com `carrega`, `params` e `modelo`,
  sendo que o Ridge sai como pipeline com `StandardScaler`. O seu docstring indica uso "nos
  notebooks de mesclagem", e a origem declarada do JSON é o `random_search.ipynb`, **não** o
  `random_search_13cat`. Nenhum notebook deste pipeline o importa.

### Observação sobre versionamento

O `.gitignore` do projeto ignora `*.csv`, `*.xls`, `*.xlsx`, `*.tsv`, `*.db` e `*.sql`, por
compromisso de proteção de dados com a parceira. Na prática:

- **nenhum** CSV do pipeline está no repositório, de modo que todos precisam ser regerados
  executando os notebooks `01` e `02`;
- o que sobrevive é o que é JSON: `particao_treino_teste.json`, `chave_replicata.json` e
  `artefatos/hiperparametros.json`;
- `src/frontend/dados/modelos/` também é ignorado, de modo que os modelos finais precisam ser
  regravados pelo `exportar.py`;
- não existe arquivo de dependências (nem `requirements.txt`, nem `pyproject.toml`, nem
  `environment.yml`), e nenhuma versão está fixada. As bibliotecas de cada notebook estão
  anotadas sob o nome dele, nas seções `01` a `07`.


---

## Célula opcional de verificação

A célula abaixo **não processa nada**: só descobre a raiz do repositório e confere, com
`pathlib`, a presença dos arquivos esperados de cada etapa. Lembre que o `.gitignore` do
projeto ignora todo `*.csv`, de modo que num clone novo **é esperado** que tudo apareça como
ausente até que os notebooks 01 e 02 sejam executados.


In [ ]:
from pathlib import Path

AREAS = ["oven_hmi", "ct_knobs", "main_vent_sys", "ipc_cooling",
         "mainback", "ipc_comp", "coil_temps"]


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios a partir de inicio ate achar a raiz do repositorio.

    Marcadores aceitos, em ordem: a pasta .git; ou a presenca simultanea de
    notebooks/ e documents/ (a mesma convencao usada pelos notebooks do projeto).
    Nao ha requirements.txt neste repositorio, de modo que ele nao serve de marcador.
    """
    atual = Path(inicio or Path.cwd()).resolve()
    for pasta in (atual, *atual.parents):
        if (pasta / ".git").exists():
            return pasta
        if (pasta / "notebooks").is_dir() and (pasta / "documents").is_dir():
            return pasta
    return None


RAIZ = raiz_repositorio()
if RAIZ is None:
    raise SystemExit("raiz do repositorio nao encontrada a partir de " + str(Path.cwd()))

S4 = RAIZ / "notebooks" / "colabs_S4"
S3P = RAIZ / "notebooks" / "colabs_S3" / "preparacao_testes"
RS = S4 / "resultados_random_search"

# Etapa -> (notebook real, [arquivos esperados])
ETAPAS = [
    ("00 fontes externas", [
        RAIZ / "documents" / "outros" / "dataset_original.xlsx",
        S3P / "particao_treino_teste.json",
        S3P / "metricas.ipynb",
    ]),
    ("01 preparacao_datasets", [S4 / "preparacao_datasets.ipynb"]
        + [S4 / f"dataset_{a}.csv" for a in AREAS]
        + [S4 / "limites_ativos.csv", S4 / "chave_replicata.json"]),
    ("02 tratamento_datasets_13cat", [S4 / "tratamento_datasets_13cat.ipynb"]
        + [S4 / f"dataset_{a}_13cat.csv" for a in AREAS]
        + [S4 / f"dataset_{a}_perfil.csv" for a in AREAS]
        + [S4 / "particao_treino_teste.csv"]),
    ("02 saidas auxiliares (_tratado)",
        [S4 / f"dataset_{a}_13cat_tratado.csv" for a in AREAS]
        + [S4 / "limites_ativos_tratado.csv"]),
    ("03 tratamento_nulos_pos_revisao", [S4 / "tratamento_nulos_pos_revisao.ipynb"]
        + [S4 / f"dataset_{a}_13cat_snull.csv" for a in AREAS]),
    ("04 modelagem", [S4 / "modelagem.ipynb"]),
    ("05 random_search_13cat", [S4 / "random_search_13cat.ipynb",
                                RS / "kpis_consolidado.csv"]),
    ("06 comparacao_modelos_final", [S4 / "comparacao_modelos_final.ipynb",
                                     S4 / "comparacao_modelos.ipynb"]),
    ("07 analises_complementares", [S4 / "analises_complementares.ipynb"]),
    ("aplicacao do modelo final", [
        RAIZ / "src" / "frontend" / "dados" / "exportar.py",
        RAIZ / "src" / "frontend" / "previsao.py",
        RAIZ / "src" / "frontend" / "dados" / "modelos" / "modelos.joblib",
    ]),
]

print("Raiz do repositorio:", RAIZ)
print()
for etapa, arquivos in ETAPAS:
    presentes = sum(1 for p in arquivos if p.exists())
    print(f"{etapa}  ({presentes}/{len(arquivos)} presentes)")
    for p in arquivos:
        marca = "ok  " if p.exists() else "FALTA"
        print(f"   {marca} {p.relative_to(RAIZ).as_posix()}")
    print()

print("Lembrete: o .gitignore do projeto ignora *.csv, *.xlsx e src/frontend/dados/modelos/.")
print("Num clone novo, os CSVs faltam ate que os notebooks 01 e 02 sejam executados,")
print("e modelos.joblib falta ate que 'python src/frontend/dados/exportar.py' seja rodado.")


---

## Conclusão sobre o modelo final utilizado

O modelo final do projeto é a **formulação entregue**, definida na **seção 23 de
`notebooks/colabs_S4/modelagem.ipynb`**. É ela que `src/frontend/dados/exportar.py` lê por `exec`
e transforma em `src/frontend/dados/modelos/modelos.joblib`, o arquivo que o dashboard consulta.

### Do que ele é feito

São **sete modelos, um por área térmica**, treinados sobre o formato longo, com uma linha por
par de ensaio e componente. A identidade do componente não entra como variável: ela é
substituída pelo **patamar** (média) e pela **escala** (desvio-padrão) das temperaturas
daquele componente, calculados **apenas nas linhas de treino de cada dobra**, de modo que o
vazamento do alvo fica impedido por construção.

A escolha do estimador e do tratamento de `losses` é feita **por área**:

| Área | Estimador | `losses` |
|---|---|---|
| `oven_hmi` | Random Forest | agrupadas em quatro níveis |
| `ct_knobs` | Random Forest | 13 categorias detalhadas |
| `main_vent_sys` | média entre Extra Trees e Gradient Boosting | 13 categorias detalhadas |
| `ipc_cooling` | Random Forest | agrupadas em quatro níveis |
| `mainback` | Random Forest | 13 categorias detalhadas |
| `ipc_comp` | Random Forest | agrupadas em quatro níveis |
| `coil_temps` | média entre Random Forest e Gradient Boosting | 13 categorias detalhadas |

Todos com `n_estimators=300` e os demais hiperparâmetros no padrão da biblioteca, semente 42.
A busca de hiperparâmetros dos notebooks `05` e `06` **não entrou na entrega**: medida com a
escolha refeita dentro de cada dobra, ela não se sustentou.

### O que mais entra no ajuste

- **Peso próximo ao limite**, `1 + 2·exp(−(d/15)²)`, com `d` a distância ao limite do
  componente: uma leitura sobre o limite pesa 3 e uma a trinta graus dele pesa praticamente 1.
  A razão é que a triagem decide perto do limite, e o erro cometido longe dele não muda
  decisão alguma.
- **Base `13cat`**, com `reason` em seis níveis mutuamente exclusivos.
- As colunas `loss_profile_order_*` ficam **fora** das entradas, por terem sido medidas como
  prejudiciais.
- Valores ausentes do alvo são **descartados por máscara**, nunca imputados. É por isso que os
  arquivos `_snull` do notebook `03` não participam.

### Quanto ele erra

| Formulação | Validação cruzada | Teste reservado |
|---|---|---|
| **Entregue** (esta) | **5,96 a 6,23 °C**, conforme o método de aninhamento | **5,154 °C** |
| Escolhendo só o estimador | 6,107 °C | 5,17 °C |
| De referência, Random Forest em todas as áreas | 6,47 °C | 5,15 °C |
| Modelo da Sprint 3, em base anterior | 6,565 °C | 4,37 °C |

A validação retira uma configuração experimental inteira por vez, em 17 dobras. Os 10 ensaios
reservados não participam de nenhuma escolha do modelo, mas foram lidos várias vezes ao longo
do projeto, o que reduz a proteção que oferecem.

### Três ressalvas que acompanham o número

1. **O teste reservado aponta em sentido contrário ao da validação.** A formulação entregue
   melhora na validação e piora no teste reservado em relação ao modelo da Sprint 3. Aquele
   modelo foi medido na base anterior, que tinha o sensor duplicado do Oven HMI e 74
   componentes contra os 73 de hoje, de modo que parte da diferença pode vir da base e não do
   modelo. A seção 20 do notebook `04` e o relatório do projeto discutem o porquê.
2. **A regra por área foi derivada olhando os resultados.** Escrita como constante, ela mede
   5,855 °C; com a escolha refeita dentro de cada dobra, mede entre 5,96 e 6,23 °C, e é essa
   faixa que o projeto reporta.
3. **O uso pretendido é de triagem, não de substituição do ensaio.** Nos notebooks, a previsão
   é comparada ao limite com margem de 8 °C, critério adotado para comparar os modelos entre
   si. Em produção vale a regra de projeto da Whirlpool, que exige folga de pelo menos 10% do
   limite: é o `FOLGA_ATENCAO_PCT = 10.0` do `exportar.py`. Em qualquer dos dois, arquiteturas
   próximas do limite continuam exigindo validação completa.


---

## Referências

Entradas conforme a ABNT, nas mesmas formas adotadas em `documents/documentacao.md`. Constam
aqui apenas as obras citadas neste guia.

BERGSTRA, James; BENGIO, Yoshua. Random search for hyper-parameter optimization. **Journal of
Machine Learning Research**, v. 13, p. 281-305, 2012.

BREIMAN, Leo. Random forests. **Machine Learning**, v. 45, n. 1, p. 5-32, 2001.

CAWLEY, Gavin C.; TALBOT, Nicola L. C. On over-fitting in model selection and subsequent
selection bias in performance evaluation. **Journal of Machine Learning Research**, v. 11,
p. 2079-2107, 2010.

FRIEDMAN, Jerome H. Greedy function approximation: a gradient boosting machine. **The Annals
of Statistics**, v. 29, n. 5, p. 1189-1232, 2001.

GEURTS, Pierre; ERNST, Damien; WEHENKEL, Louis. Extremely randomized trees. **Machine
Learning**, v. 63, n. 1, p. 3-42, 2006.

KAUFMAN, Shachar; ROSSET, Saharon; PERLICH, Claudia; STITELMAN, Ori. Leakage in data mining:
formulation, detection, and avoidance. **ACM Transactions on Knowledge Discovery from Data**,
v. 6, n. 4, artigo 15, 2012.

LUNDBERG, Scott M.; LEE, Su-In. A unified approach to interpreting model predictions. In:
CONFERENCE ON NEURAL INFORMATION PROCESSING SYSTEMS, 31., 2017, Long Beach. **Advances in
Neural Information Processing Systems**. Red Hook: Curran Associates, 2017. p. 4765-4774.

PEDREGOSA, Fabian et al. Scikit-learn: machine learning in Python. **Journal of Machine
Learning Research**, v. 12, p. 2825-2830, 2011.
